# EDA — all_games (Analítica Avanzada)

Análisis exploratorio (EDA) del dataset `data/all_games.csv` con gráficos modernos (Plotly).

## Objetivo
- Entender la estructura del dataset.
- Revisar calidad de datos (nulos, duplicados, tipos).
- Generar visualizaciones modernas para detectar patrones.

**Archivo:** `data/all_games.csv`
**Columnas:** `name`, `platform`, `release_date`, `summary`, `meta_score`, `user_review`

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 120)


In [ ]:
df = pd.read_csv('data/all_games.csv', low_memory=False)
print('Shape:', df.shape)
df.head(10)

In [ ]:
df.info()

missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing': missing, 'missing_%': missing_pct})

In [ ]:
dup_count = df.duplicated().sum()
print('Duplicados:', dup_count)

# Limpieza mínima (sin modificar el CSV): quitar duplicados exactos
df = df.drop_duplicates().copy()
print('Shape después de drop_duplicates:', df.shape)

# Conversión robusta de tipos
df['meta_score'] = pd.to_numeric(df['meta_score'], errors='coerce')
df['user_review'] = pd.to_numeric(df['user_review'], errors='coerce')
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')

df[['meta_score', 'user_review', 'release_date']].describe(include='all')

In [ ]:
# Feature engineering: año de lanzamiento
df['release_year'] = df['release_date'].dt.year
df[['release_date', 'release_year']].head()

## Gráficos modernos (Plotly)
Los gráficos son interactivos: zoom, selección y tooltips.

In [ ]:
# Top plataformas por número de juegos
platform_counts = (
    df['platform']
      .astype('string')
      .fillna('Unknown')
      .value_counts()
      .head(20)
      .reset_index()
)
platform_counts.columns = ['platform', 'count']

fig = px.bar(
    platform_counts,
    x='platform',
    y='count',
    title='Top 20 plataformas (cantidad de juegos)',
    template='plotly_white'
)
fig.update_layout(xaxis_title='Plataforma', yaxis_title='Cantidad', xaxis_tickangle=-45)
fig.show()

In [ ]:
# Distribución de meta_score
fig = px.histogram(
    df,
    x='meta_score',
    nbins=30,
    title='Distribución de Meta Score',
    template='plotly_white'
)
fig.update_layout(xaxis_title='Meta Score', yaxis_title='Frecuencia')
fig.show()

In [ ]:
# Distribución de user_review (solo valores numéricos válidos)
fig = px.histogram(
    df,
    x='user_review',
    nbins=30,
    title='Distribución de User Review',
    template='plotly_white'
)
fig.update_layout(xaxis_title='User Review', yaxis_title='Frecuencia')
fig.show()

In [ ]:
# Relación: Meta Score vs User Review
plot_df = df[['name', 'platform', 'meta_score', 'user_review']].dropna(subset=['meta_score', 'user_review']).copy()

fig = px.scatter(
    plot_df,
    x='meta_score',
    y='user_review',
    color='platform',
    hover_name='name',
    title='Meta Score vs User Review (por plataforma)',
    template='plotly_white'
)
fig.update_layout(xaxis_title='Meta Score', yaxis_title='User Review')
fig.show()

In [ ]:
# Tendencia temporal: cantidad de juegos por año
year_counts = (
    df.dropna(subset=['release_year'])
      .groupby('release_year')
      .size()
      .reset_index(name='count')
      .sort_values('release_year')
)

fig = px.line(
    year_counts,
    x='release_year',
    y='count',
    title='Juegos por año de lanzamiento',
    template='plotly_white'
)
fig.update_layout(xaxis_title='Año', yaxis_title='Cantidad')
fig.show()

## Siguientes pasos (sugeridos)
- Revisar outliers en `meta_score` y `user_review`.
- Analizar disponibilidad de reseñas (nulos vs no nulos).
- Comparar por plataforma y por rango de años.

## Outliers (IQR)
Detectamos outliers en `meta_score` y `user_review` usando el criterio IQR (rango intercuartílico).

In [ ]:
def iqr_bounds(series: pd.Series):
    s = pd.to_numeric(series, errors='coerce').dropna()
    if s.empty:
        return np.nan, np.nan
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ['meta_score', 'user_review']:
    low, high = iqr_bounds(df[col])
    out = df[(df[col] < low) | (df[col] > high)][['name', 'platform', 'release_year', col]].dropna(subset=[col])
    print(f'\n{col}: límites IQR -> low={low:.3f}, high={high:.3f} | outliers={len(out)}')
    display(out.sort_values(col, ascending=False).head(10))
    display(out.sort_values(col, ascending=True).head(10))

In [ ]:
# Boxplots para visualizar outliers
fig = px.box(df, y='meta_score', points='outliers', title='Boxplot — Meta Score', template='plotly_white')
fig.show()

fig = px.box(df, y='user_review', points='outliers', title='Boxplot — User Review', template='plotly_white')
fig.show()

## Disponibilidad de scores y reseñas
Calculamos qué porcentaje de juegos tiene `meta_score` y `user_review` disponibles.

In [ ]:
availability = pd.DataFrame({
    'columna': ['meta_score', 'user_review'],
    'no_nulos': [df['meta_score'].notna().sum(), df['user_review'].notna().sum()],
})
availability['nulos'] = len(df) - availability['no_nulos']
availability['no_nulos_%'] = (availability['no_nulos'] / len(df) * 100).round(2)
availability['nulos_%'] = (availability['nulos'] / len(df) * 100).round(2)
availability

fig = px.bar(
    availability,
    x='columna',
    y='no_nulos_%',
    title='Porcentaje de valores disponibles',
    text='no_nulos_%',
    template='plotly_white'
)
fig.update_layout(xaxis_title='Variable', yaxis_title='% disponible')
fig.show()

## Comparación por plataforma (Top 10)
Comparamos distribuciones de `meta_score` y `user_review` en las 10 plataformas con más juegos.

In [ ]:
top_platforms = df['platform'].astype('string').fillna('Unknown').value_counts().head(10).index
sub = df[df['platform'].astype('string').fillna('Unknown').isin(top_platforms)].copy()
sub['platform'] = sub['platform'].astype('string').fillna('Unknown')

fig = px.box(
    sub.dropna(subset=['meta_score']),
    x='platform',
    y='meta_score',
    points='outliers',
    title='Meta Score por plataforma (Top 10)',
    template='plotly_white'
)
fig.update_layout(xaxis_title='Plataforma', yaxis_title='Meta Score', xaxis_tickangle=-45)
fig.show()

fig = px.box(
    sub.dropna(subset=['user_review']),
    x='platform',
    y='user_review',
    points='outliers',
    title='User Review por plataforma (Top 10)',
    template='plotly_white'
)
fig.update_layout(xaxis_title='Plataforma', yaxis_title='User Review', xaxis_tickangle=-45)
fig.show()

## Comparación temporal (por año)
Analizamos la evolución temporal de la cantidad de juegos y los promedios de score (cuando existen).

In [ ]:
year_stats = (
    df.dropna(subset=['release_year'])
      .groupby('release_year')
      .agg(
          juegos=('name', 'size'),
          meta_score_mean=('meta_score', 'mean'),
          user_review_mean=('user_review', 'mean')
      )
      .reset_index()
      .sort_values('release_year')
)

fig = px.line(
    year_stats,
    x='release_year',
    y='juegos',
    title='Cantidad de juegos por año',
    template='plotly_white'
)
fig.update_layout(xaxis_title='Año', yaxis_title='Juegos')
fig.show()

mean_long = year_stats.melt(
    id_vars=['release_year'],
    value_vars=['meta_score_mean', 'user_review_mean'],
    var_name='métrica',
    value_name='promedio'
)
mean_long['métrica'] = mean_long['métrica'].replace({
    'meta_score_mean': 'Meta Score (promedio)',
    'user_review_mean': 'User Review (promedio)'
})

fig = px.line(
    mean_long.dropna(subset=['promedio']),
    x='release_year',
    y='promedio',
    color='métrica',
    title='Promedios de score por año (cuando hay datos)',
    template='plotly_white'
)
fig.update_layout(xaxis_title='Año', yaxis_title='Promedio')
fig.show()